# 신경망 텐서의 메모리 레이아웃 - Row-Major vs Column-Major와 캐시 효율성

- Tutorial ID: `cs-1-1`
- Tutorial: 신경망 텐서의 메모리 레이아웃
- Section ID: `cs-1-1-1`
- Section: Row-Major vs Column-Major와 캐시 효율성


# 060 | 신경망 텐서의 메모리 레이아웃
## Row-Major vs Column-Major와 캐시 효율성

> **대상**: 딥러닝 코드를 짰는데 왜 느린지 궁금한 분, 메모리 최적화에 관심 있는 분  
> **목표**: 컴퓨터 메모리의 구조를 이해하고, 텐서 연산의 캐시 효율성을 실습으로 체험한다.

---

## 📌 목차
1. 컴퓨터 메모리 구조: 캐시란 무엇인가?
2. Row-Major vs Column-Major 레이아웃
3. NumPy 메모리 레이아웃 확인하기
4. 캐시 효율성과 성능: 실제 측정
5. 트랜스포머에서의 메모리 레이아웃
6. contiguous 메모리와 transpose의 함정
7. 실용적인 최적화 팁

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib
import sys

try:
    matplotlib.rc('font', family='NanumGothic')
except:
    pass
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)

print(f'Python {sys.version}')
print(f'NumPy {np.__version__}')
print('라이브러리 로드 완료!')

---
## 1. 컴퓨터 메모리 구조: 캐시란?

### 메모리 계층 구조

컴퓨터는 데이터를 저장하고 접근하는 속도가 다른 여러 단계의 메모리를 가집니다:

```
빠름 ↑  CPU 레지스터  (수 바이트,    < 1ns)
      │  L1 캐시      (32-64 KB,    ~1ns)
      │  L2 캐시      (256 KB-1 MB, ~4ns)
      │  L3 캐시      (4-64 MB,    ~10ns)
느림 ↓  RAM (주메모리) (수 GB,     ~100ns)
         SSD          (수 TB,      ~100μs)
```

### 캐시 동작 원리

CPU가 메모리에서 데이터 하나를 읽을 때, **인접한 데이터를 함께** 캐시에 올립니다.  
이것을 **캐시 라인(Cache Line)**이라 하며, 보통 **64 바이트** (float32 기준 16개)입니다.

```
메모리에서 A[0]를 읽으면 → A[0]~A[15]가 캐시에 올라옴
다음에 A[1]을 읽을 때 → 이미 캐시에 있음! (Cache Hit → 빠름)
하지만 A[100]을 읽으면 → 캐시에 없음 (Cache Miss → 느림, 메모리 접근)
```

**핵심 원칙:** 메모리를 **순서대로(연속적으로)** 접근하면 캐시 적중률이 높아집니다!

In [ ]:
# ============================================================
# 캐시 효과 직관적 데모

In [ ]:
print('=== 순차 접근 vs 랜덤 접근 속도 비교 ===')
print()

size = 1_000_000  # 100만개 원소
arr = np.random.randn(size).astype(np.float32)

# 순차 접근: arr[0], arr[1], arr[2], ...
n_trials = 5
sequential_times = []
for _ in range(n_trials):
    start = time.perf_counter()
    total = arr.sum()  # 순차적으로 전체를 더함
    sequential_times.append(time.perf_counter() - start)

# 랜덤 접근: 랜덤한 인덱스로 접근
random_idx = np.random.permutation(size)
random_times = []
for _ in range(n_trials):
    start = time.perf_counter()
    total = arr[random_idx].sum()  # 랜덤 인덱스로 접근
    random_times.append(time.perf_counter() - start)

seq_mean = np.mean(sequential_times) * 1000
rand_mean = np.mean(random_times) * 1000

print(f'배열 크기: {size:,}개 float32 ({size*4/1e6:.1f} MB)')
print(f'순차 접근 평균 시간: {seq_mean:.3f} ms')
print(f'랜덤 접근 평균 시간: {rand_mean:.3f} ms')
print(f'속도 차이: {rand_mean/seq_mean:.1f}배 (랜덤이 더 느림)')
print()
print('▶ 순차 접근이 더 빠른 이유:')
print('  1. 캐시 라인 효율적 사용 (순서대로 읽으면 캐시에 미리 올라옴)')
print('  2. CPU의 Pre-fetcher가 다음 데이터를 미리 가져옴')

---
## 2. Row-Major vs Column-Major 레이아웃

2D 행렬을 1D 메모리에 저장할 때 두 가지 방법이 있습니다:

### Row-Major (행 우선, C 스타일) — NumPy 기본
```
행렬:  [A B C]      메모리: A B C D E F  (왼쪽→오른쪽, 위→아래)
       [D E F]             ↑ 행 순서대로 저장
```

### Column-Major (열 우선, Fortran/MATLAB 스타일)
```
행렬:  [A B C]      메모리: A D B E C F  (위→아래, 왼쪽→오른쪽)
       [D E F]             ↑ 열 순서대로 저장
```

**PyTorch와 NumPy는 Row-Major (C-contiguous)를 기본으로 사용합니다.**

In [ ]:
# ============================================================
# Row-Major vs Column-Major 시각화

In [ ]:
print('=== Row-Major vs Column-Major 메모리 배치 ===')
print()

# 3x4 행렬 예제
matrix = np.array([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
])

print('원본 행렬:')
print(matrix)
print(f'shape: {matrix.shape}')

# Row-Major (C order): 행 우선
row_major = matrix.flatten(order='C')  # 기본값
print(f'\nRow-Major (C order) 메모리 배치:')
print(row_major)
print('→ 1행을 다 저장 후 2행, 3행 순서')

# Column-Major (Fortran order): 열 우선
col_major = matrix.flatten(order='F')
print(f'\nColumn-Major (Fortran order) 메모리 배치:')
print(col_major)
print('→ 1열을 다 저장 후 2열, 3열 순서')

print()
# NumPy 배열의 strides 확인
# strides: 다음 원소로 이동하기 위한 바이트 수
arr_c = np.array(matrix, order='C')  # Row-Major
arr_f = np.array(matrix, order='F')  # Column-Major

print('=== Strides (메모리 이동 단계) ===')
print(f'Row-Major strides:    {arr_c.strides} bytes')
print(f'  → 다음 행으로: {arr_c.strides[0]} bytes ({arr_c.strides[0]//4}개 float32 건너뜀)')
print(f'  → 다음 열로:   {arr_c.strides[1]} bytes ({arr_c.strides[1]//4}개 float32 건너뜀)')
print()
print(f'Col-Major strides:    {arr_f.strides} bytes')
print(f'  → 다음 행으로: {arr_f.strides[0]} bytes ({arr_f.strides[0]//4}개 float32 건너뜀)')
print(f'  → 다음 열로:   {arr_f.strides[1]} bytes ({arr_f.strides[1]//4}개 float32 건너뜀)')

In [ ]:
# ============================================================
# Strides 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3x4 행렬의 원소를 색상으로 표현
data = np.arange(12).reshape(3, 4)

# Row-Major 순서 시각화
axes[0].set_title('Row-Major (C order)\nNumPy/PyTorch 기본', fontsize=12)

# 각 원소의 메모리 인덱스 (C order)
for i in range(3):
    for j in range(4):
        mem_idx = i * 4 + j  # Row-Major 메모리 인덱스
        color = plt.cm.RdYlGn(mem_idx / 12)
        axes[0].add_patch(plt.Rectangle((j, 2-i), 1, 1, color=color, ec='black', lw=2))
        axes[0].text(j+0.5, 2-i+0.5, f'[{i},{j}]\n메모리#{mem_idx}', 
                    ha='center', va='center', fontsize=9)

axes[0].set_xlim(0, 4)
axes[0].set_ylim(0, 3)
axes[0].set_xlabel('열 (column) →')
axes[0].set_ylabel('← 행 (row)')
axes[0].set_xticks(range(5))
axes[0].set_yticks(range(4))

# Column-Major 순서 시각화
axes[1].set_title('Column-Major (Fortran order)\nMATLAB/BLAS 사용', fontsize=12)

for i in range(3):
    for j in range(4):
        mem_idx = j * 3 + i  # Column-Major 메모리 인덱스
        color = plt.cm.RdYlGn(mem_idx / 12)
        axes[1].add_patch(plt.Rectangle((j, 2-i), 1, 1, color=color, ec='black', lw=2))
        axes[1].text(j+0.5, 2-i+0.5, f'[{i},{j}]\n메모리#{mem_idx}',
                    ha='center', va='center', fontsize=9)

axes[1].set_xlim(0, 4)
axes[1].set_ylim(0, 3)
axes[1].set_xlabel('열 (column) →')
axes[1].set_ylabel('← 행 (row)')
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(4))

# 범례
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(0, 12))
plt.colorbar(sm, ax=axes, label='메모리 주소 순서 (작을수록 먼저 저장)')

plt.suptitle('3×4 행렬의 메모리 저장 방식 비교', fontsize=13)
plt.tight_layout()
plt.show()

print('▶ Row-Major: 같은 행의 원소들이 메모리에서 인접 (행 접근 유리)')
print('▶ Col-Major: 같은 열의 원소들이 메모리에서 인접 (열 접근 유리)')

---
## 3. NumPy 메모리 레이아웃 확인하기

In [ ]:
# ============================================================
# NumPy 배열의 메모리 정보 확인

In [ ]:
print('=== NumPy 배열 메모리 정보 ===')
print()

# C-contiguous (Row-Major) 배열
arr_c = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
print('C-contiguous 배열 (기본):')
print(f'  배열: {arr_c}')
print(f'  shape: {arr_c.shape}')
print(f'  strides: {arr_c.strides}  (행 방향: {arr_c.strides[0]}B, 열 방향: {arr_c.strides[1]}B)')
print(f'  C-contiguous: {arr_c.flags["C_CONTIGUOUS"]}')
print(f'  F-contiguous: {arr_c.flags["F_CONTIGUOUS"]}')
print()

# Transpose: 뷰(view)를 반환 — 실제 메모리는 변하지 않음!
arr_t = arr_c.T  # arr_c의 '뷰' (복사가 아님!)
print('Transpose 후:')
print(f'  배열: {arr_t}')
print(f'  shape: {arr_t.shape}')
print(f'  strides: {arr_t.strides}  (음수 없음 — 단순히 strides 교환!)')
print(f'  C-contiguous: {arr_t.flags["C_CONTIGUOUS"]}')
print(f'  F-contiguous: {arr_t.flags["F_CONTIGUOUS"]}')
print(f'  → Transpose는 데이터를 복사하지 않고 strides만 바꿉니다!')
print()

# 같은 메모리를 공유하는지 확인
print(f'arr_c와 arr_t가 같은 메모리 공유? {np.shares_memory(arr_c, arr_t)}')
print('▶ 공유하면 arr_t를 수정하면 arr_c도 변합니다!')
arr_t[0, 0] = 999
print(f'arr_t[0,0] = 999 변경 후 arr_c[0,0]: {arr_c[0,0]}  (arr_c도 변함!)')

In [ ]:
# ============================================================
# contiguous 복사와 비용

In [ ]:
print('=== contiguous 복사 비용 ===')
print()

# np.ascontiguousarray(): non-contiguous를 contiguous로 변환
arr = np.random.randn(1000, 1000).astype(np.float32)
arr_t = arr.T  # non-contiguous (F-order 처럼 보임)

print(f'원본 arr: C-contiguous={arr.flags["C_CONTIGUOUS"]}')
print(f'전치 arr.T: C-contiguous={arr_t.flags["C_CONTIGUOUS"]}')

# contiguous 복사 시간
n = 100
t1 = time.perf_counter()
for _ in range(n):
    arr_cont = np.ascontiguousarray(arr_t)  # 복사 발생
t2 = time.perf_counter()
copy_time = (t2 - t1) / n * 1000

# 이미 contiguous인 경우 (복사 없음)
t3 = time.perf_counter()
for _ in range(n):
    arr_cont2 = np.ascontiguousarray(arr)  # 이미 contiguous → 복사 안 함
t4 = time.perf_counter()
no_copy_time = (t4 - t3) / n * 1000

print(f'\nnp.ascontiguousarray(arr.T) 시간: {copy_time:.3f} ms (복사 발생)')
print(f'np.ascontiguousarray(arr) 시간:   {no_copy_time:.4f} ms (복사 안 함)')
print(f'속도 차이: {copy_time/no_copy_time:.0f}배')
print()
print('▶ PyTorch의 .contiguous() 호출도 동일한 비용이 발생합니다!')
print('  tensor.view()는 contiguous를 요구하고,')
print('  tensor.reshape()은 필요하면 자동으로 복사합니다.')

---
## 4. 캐시 효율성과 성능: 행 vs 열 반복

In [ ]:
# ============================================================
# 행 방향 vs 열 방향 접근 속도 비교

In [ ]:
print('=== 행 vs 열 방향 접근 속도 비교 ===')
print()
print('Row-Major 배열에서:')
print('  - 행 방향 접근 → 연속된 메모리 → 캐시 효율 높음 (빠름)')
print('  - 열 방향 접근 → 메모리 건너뜀 → 캐시 효율 낮음 (느림)')
print()

N = 2000  # NxN 행렬
mat = np.random.randn(N, N).astype(np.float32)
n_trials = 3

# 행 방향 합산: 각 행의 합
row_times = []
for _ in range(n_trials):
    t = time.perf_counter()
    result = mat.sum(axis=1)  # 각 행의 합 (연속 메모리 접근)
    row_times.append(time.perf_counter() - t)

# 열 방향 합산: 각 열의 합  
col_times = []
for _ in range(n_trials):
    t = time.perf_counter()
    result = mat.sum(axis=0)  # 각 열의 합 (비연속 메모리 접근)
    col_times.append(time.perf_counter() - t)

row_ms = np.mean(row_times) * 1000
col_ms = np.mean(col_times) * 1000

print(f'{N}x{N} float32 행렬 ({N*N*4/1e6:.1f} MB):')
print(f'  행 방향 합산 (axis=1): {row_ms:.3f} ms')
print(f'  열 방향 합산 (axis=0): {col_ms:.3f} ms')
print(f'  속도 비율: {col_ms/row_ms:.2f}x')
print()

# 행렬 곱셈에서의 레이아웃 효과
print('=== 행렬 곱셈에서의 레이아웃 효과 ===')
A = np.random.randn(N, N).astype(np.float32)
B = np.random.randn(N, N).astype(np.float32)

# C-contiguous vs F-contiguous
B_c = np.ascontiguousarray(B)       # C-order
B_f = np.asfortranarray(B)          # F-order (전치된 것을 다시 전치)

times_c = []
for _ in range(3):
    t = time.perf_counter()
    _ = A @ B_c
    times_c.append(time.perf_counter() - t)

times_f = []
for _ in range(3):
    t = time.perf_counter()
    _ = A @ B_f
    times_f.append(time.perf_counter() - t)

print(f'A @ B (C-contiguous): {np.mean(times_c)*1000:.1f} ms')
print(f'A @ B (F-contiguous): {np.mean(times_f)*1000:.1f} ms')
print(f'→ NumPy의 BLAS 내부에서 레이아웃을 감지하고 최적화합니다')

---
## 5. 트랜스포머에서의 메모리 레이아웃

### 트랜스포머 데이터 흐름과 레이아웃

```
입력 텐서 X: (batch=B, seq=T, d_model=D)

메모리 배치 (C-contiguous):
  X[0,0,0], X[0,0,1], ..., X[0,0,D-1],  ← 배치0, 토큰0의 임베딩 (연속!)
  X[0,1,0], X[0,1,1], ..., X[0,1,D-1],  ← 배치0, 토큰1의 임베딩
  ...
  X[B-1,T-1,0], ..., X[B-1,T-1,D-1]    ← 마지막 배치, 마지막 토큰
```

이렇게 하면 **같은 토큰의 임베딩 차원**들이 메모리에서 연속적입니다.  
→ W_Q, W_K, W_V 곱셈 시 **행렬의 열 접근**이 연속적 = 캐시 효율적!

In [ ]:
# ============================================================
# 트랜스포머 텐서 레이아웃 분석

In [ ]:
print('=== 트랜스포머 텐서 레이아웃 분석 ===')
print()

# GPT-2 small 기준 설정
B = 8     # 배치 크기
T = 512   # 시퀀스 길이
D = 768   # d_model (GPT-2 small)

# 텐서 메모리 크기 계산
tensor_bytes = B * T * D * 4  # float32 = 4 bytes
print(f'GPT-2 Small 기준:')
print(f'  입력 텐서 X: ({B}, {T}, {D})')
print(f'  메모리 크기: {tensor_bytes / 1e6:.1f} MB')
print()

# 작은 버전으로 실제 레이아웃 확인
B_, T_, D_ = 2, 4, 8
X_demo = np.arange(B_*T_*D_).reshape(B_, T_, D_)

print(f'데모 텐서 X: shape={X_demo.shape}')
print(f'strides: {X_demo.strides}  (배치:{X_demo.strides[0]}B, 시퀀스:{X_demo.strides[1]}B, 특성:{X_demo.strides[2]}B)')
print()
print('X[0, :, :] (배치0의 모든 토큰):')
print(X_demo[0])
print()
print('X[0, 0, :] (배치0, 토큰0의 임베딩 — 메모리에서 연속!):')
print(X_demo[0, 0, :])
print()
print('X[0, :, 0] (배치0의 모든 토큰의 첫번째 특성 — 메모리에서 비연속!):')
print(X_demo[0, :, 0])
print(f'  → 다음 원소까지의 간격: {X_demo.strides[1]//4}개 원소 (={D_})')

In [ ]:
# ============================================================
# Multi-Head Attention에서의 transpose 함정

In [ ]:
print('=== Multi-Head Attention의 Transpose 비용 ===')
print()

B, T, D = 4, 64, 256
H = 8  # heads
d_k = D // H  # = 32

# Q 텐서: (B, T, D)
Q = np.random.randn(B, T, D).astype(np.float32)

print(f'Q: {Q.shape}, C-contiguous: {Q.flags["C_CONTIGUOUS"]}')

# reshape: (B, T, H, d_k)
Q_reshaped = Q.reshape(B, T, H, d_k)
print(f'reshape 후: {Q_reshaped.shape}, C-contiguous: {Q_reshaped.flags["C_CONTIGUOUS"]}')

# transpose: (B, H, T, d_k) — 이때 non-contiguous 발생!
Q_transposed = Q_reshaped.transpose(0, 2, 1, 3)
print(f'transpose 후: {Q_transposed.shape}, C-contiguous: {Q_transposed.flags["C_CONTIGUOUS"]}')
print(f'  → non-contiguous! strides: {Q_transposed.strides}')

print()
# 이후 연산에서 비효율 발생
# contiguous로 만들면 복사 비용 발생하지만 이후 연산이 빨라질 수 있음
t1 = time.perf_counter()
for _ in range(50):
    # non-contiguous로 행렬곱
    score = Q_transposed @ Q_transposed.transpose(0, 1, 3, 2)
t2 = time.perf_counter()
non_cont_time = (t2 - t1) / 50 * 1000

Q_cont = np.ascontiguousarray(Q_transposed)  # 복사
t3 = time.perf_counter()
for _ in range(50):
    # contiguous로 행렬곱
    score = Q_cont @ Q_cont.transpose(0, 1, 3, 2)
t4 = time.perf_counter()
cont_time = (t4 - t3) / 50 * 1000

print(f'Non-contiguous Q @ Q.T: {non_cont_time:.2f} ms')
print(f'Contiguous Q @ Q.T:     {cont_time:.2f} ms')
print()
print('▶ PyTorch에서는 이 문제를 해결하기 위해')
print('  - F.scaled_dot_product_attention()이 내부적으로 최적화합니다')
print('  - Flash Attention은 메모리 레이아웃을 타일 기반으로 최적화합니다')

---
## 6. 실용적인 최적화 팁

In [ ]:
# ============================================================
# 실용적인 메모리 최적화 팁

In [ ]:
print('=== 실용적인 최적화 팁 ===')
print()

# 팁 1: dtype 선택
print('[ 팁 1: dtype에 따른 메모리와 속도 차이 ]')
N = 1000
dtypes = [np.float64, np.float32, np.float16]
dtype_names = ['float64', 'float32', 'float16']

for dtype, name in zip(dtypes, dtype_names):
    A = np.random.randn(N, N).astype(dtype)
    mem_mb = A.nbytes / 1e6
    
    t = time.perf_counter()
    for _ in range(5):
        _ = A @ A
    elapsed = (time.perf_counter() - t) / 5 * 1000
    
    print(f'  {name}: 메모리 {mem_mb:.1f} MB, 행렬곱 {elapsed:.1f} ms')

print()
print('  → float32가 딥러닝에서 가장 많이 쓰임 (속도/정밀도 균형)')
print('  → float16/bfloat16은 메모리 절반 + GPU에서 빠름 (혼합 정밀도)')

print()
# 팁 2: 연산 순서 최적화
print('[ 팁 2: 연산 순서 최적화 ]')
print('  (A @ B) @ C vs A @ (B @ C) — 어느 쪽이 빠를까?')
print()

# 행렬 크기: A(1000x10), B(10x1000), C(1000x10)
A_m = np.random.randn(1000, 10).astype(np.float32)
B_m = np.random.randn(10, 1000).astype(np.float32)
C_m = np.random.randn(1000, 10).astype(np.float32)

# 방법 1: (A@B)@C → (1000x10)@(10x1000) = 1000x1000, then @C → 1000x10
#          연산: 1000*1000*10 + 1000*1000*10 = 20,000,000 곱셈
t1 = time.perf_counter()
for _ in range(20):
    r1 = (A_m @ B_m) @ C_m
t2 = time.perf_counter()

# 방법 2: A@(B@C) → (10x1000)@(1000x10) = 10x10, then A@ → 1000x10
#          연산: 10*10*1000 + 1000*10*10 = 200,000 곱셈 (100배 적음!)
t3 = time.perf_counter()
for _ in range(20):
    r2 = A_m @ (B_m @ C_m)
t4 = time.perf_counter()

time1 = (t2 - t1) / 20 * 1000
time2 = (t4 - t3) / 20 * 1000

print(f'  (A@B)@C 시간: {time1:.2f} ms  [연산량: 20,000,000]')
print(f'  A@(B@C) 시간: {time2:.2f} ms  [연산량:    200,000]')
print(f'  속도 차이: {time1/time2:.1f}배')
print()
print('  → 행렬 곱셈 순서가 중요합니다! 결합법칙은 성립하지만 연산량이 다릅니다.')
print('  → 트랜스포머에서 FFN: (X @ W1) @ W2 vs X @ (W1 @ W2)')
print('     일반적으로 (X @ W1) @ W2가 더 효율적 (중간 행렬이 더 작음)')

In [ ]:
# ============================================================
# 최적화 요약: PyTorch 스타일 가이드

In [ ]:
print('=' * 55)
print('  메모리 레이아웃 최적화 체크리스트 (PyTorch 기준)')
print('=' * 55)

tips = [
    ('✅', 'tensor.contiguous() 필요 시 명시적 호출',
     'view() 전에 반드시 contiguous 확인'),
    ('✅', 'reshape() 대신 view() 사용 시 주의',
     'view()는 contiguous 텐서만 허용'),
    ('✅', 'Batch dimension을 첫번째로',
     '(B, T, D) 형태가 대부분 최적'),
    ('✅', 'transpose 후 연산 많으면 .contiguous() 고려',
     '복사 비용 vs 연산 효율 트레이드오프'),
    ('✅', 'float32로 학습, 추론 시 float16/bfloat16 고려',
     '메모리 2배 절약, GPU Tensor Core 활용'),
    ('✅', 'torch.compile() 활용 (PyTorch 2.0+)',
     '자동으로 메모리 접근 패턴 최적화'),
    ('✅', 'Flash Attention 사용 (transformers 라이브러리)',
     '타일링으로 HBM 접근 최소화'),
]

for icon, tip, detail in tips:
    print(f'\n{icon} {tip}')
    print(f'   └─ {detail}')

---
## 📚 핵심 정리

| 개념 | 설명 | 딥러닝 관련성 |
|------|------|---------------|
| **캐시** | 빠른 임시 메모리 | 연속 메모리 접근이 캐시 히트 |
| **Row-Major** | 행 우선 저장 (NumPy/PyTorch 기본) | X[b, t, :] 접근이 효율적 |
| **Strides** | 각 차원의 메모리 이동 바이트 | Transpose = strides만 교환 |
| **Contiguous** | 메모리가 연속된 배열 | view/reshape 필요 조건 |
| **dtype** | float16 < float32 < float64 | 메모리/속도/정밀도 트레이드오프 |
| **Flash Attention** | 타일 기반 Attention | HBM 접근 최소화로 GPU 효율화 |

### 다음 단계
- **061**: GPU 아키텍처와 트랜스포머 연산
- **062**: FP16/BF16 혼합 정밀도 학습